# 02 Provenance, schema and labels (Phase 2)
Produced by `python -I src/audit/schema_labels.py scan` and `derive`. This notebook re-reads every header directly and checks it against the saved schema.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = ROOT / "results/audit"
M = ROOT / "data/manifests"
pd.set_option("display.max_rows", 80); pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("project root:", ROOT.name)

project root: research_paper


In [2]:
S = json.loads((R / 'schema_by_file.json').read_text(encoding='utf-8'))['files']
pd.DataFrame([{k: s[k] for k in ['relpath','n_columns','n_rows','read_mode','source_encoding','line_ending']}
              | {'lines_not_read': s['newline_check']['lines_not_read_as_rows']} for s in S])

,relpath,n_columns,n_rows,read_mode,source_encoding,line_ending,lines_not_read
0,data/raw_improved/CIC-IDS-2017/friday.csv,91,547557,strict,utf-8,LF,0
1,data/raw_improved/CIC-IDS-2017/monday.csv,91,371624,strict,utf-8,LF,0
2,data/raw_improved/CIC-IDS-2017/thursday.csv,91,362076,strict,utf-8,LF,0
3,data/raw_improved/CIC-IDS-2017/tuesday.csv,91,322078,strict,utf-8,LF,0
4,data/raw_improved/CIC-IDS-2017/wednesday.csv,91,496641,strict,utf-8,LF,0
5,data/raw_improved/CSE-CIC-IDS-2018/Friday-02-0...,91,6311371,strict,utf-8,LF,0
6,data/raw_improved/CSE-CIC-IDS-2018/Friday-16-0...,91,7390266,strict,utf-8,LF,0
7,data/raw_improved/CSE-CIC-IDS-2018/Friday-23-0...,91,5976481,strict,utf-8,LF,0
8,data/raw_improved/CSE-CIC-IDS-2018/Thursday-01...,91,6551401,strict,utf-8,LF,0
9,data/raw_improved/CSE-CIC-IDS-2018/Thursday-15...,91,5410102,strict,utf-8,LF,0


In [3]:
# independent check: exact header names re-read from disk == saved schema
from src.audit.common import read_header
bad = [s['relpath'] for s in S if read_header(ROOT / s['relpath'])['names'] != s['header_names_exact']]
print('header mismatches:', bad)

header mismatches: []


In [4]:
L = pd.read_csv(R / 'label_distribution_by_dataset.csv', keep_default_na=False)
L[L.variant == 'improved'][['dataset_family','label','attempted_category','count','proportion']]

,dataset_family,label,attempted_category,count,proportion
0,CIC-IDS-2017,BENIGN,-1,1582566,7.536115e-01
1,CIC-IDS-2017,Portscan,-1,159066,7.574658e-02
2,CIC-IDS-2017,DoS Hulk,-1,158468,7.546181e-02
3,CIC-IDS-2017,DDoS,-1,95144,4.530718e-02
4,CIC-IDS-2017,Infiltration - Portscan,-1,71767,3.417515e-02
5,CIC-IDS-2017,DoS GoldenEye,-1,7567,3.603375e-03
6,CIC-IDS-2017,Botnet - Attempted,1,4067,1.936689e-03
7,CIC-IDS-2017,FTP-Patator,-1,3972,1.891450e-03
8,CIC-IDS-2017,DoS Slowloris,-1,3859,1.837640e-03
9,CIC-IDS-2017,SSH-Patator,-1,2961,1.410016e-03


In [5]:
pd.read_csv(R / 'label_vs_labelling_rules.csv').groupby(['dataset_family','status']).size()

dataset_family    status                               
CIC-IDS-2017      benign_default (label_rest_as_benign)     5
                  rule_and_data                            38
                  rule_matched_no_flows                     7
CSE-CIC-IDS-2018  benign_default (label_rest_as_benign)    10
                  rule_and_data                            47
                  rule_matched_no_flows                    11
dtype: int64

In [6]:
pd.read_csv(R / 'schema_diff_original_vs_improved.csv').groupby(['dataset_family','status']).size()

dataset_family    status   
CIC-IDS-2017      added         7
                  removed       1
                  renamed      73
                  unchanged    11
CSE-CIC-IDS-2018  added         7
                  renamed      48
                  unchanged    36
dtype: int64